# Original FINet baseline training
Uses torchvision ImageNet-pretrained EfficientNet-B0 and the original FINet modules, DCT pipeline and four-output structure loss. The heads start randomly. A smoke checkpoint is not a successful camouflage detector.

Choose **Runtime → Change runtime type → GPU** (free availability is not guaranteed). Run cells in order. The full-training cell is disabled by default; inspect the measured GPU smoke report and configuration before deliberately enabling it. This baseline uses boundary_lambda=0; the optional boundary experiment is configured separately.

Upload `finet_colab_training.zip` generated by `python scripts/package_colab.py` from the current workspace. Upstream FINet alone does not contain these additions. The dataset helper below obtains the official training archive; see `docs/COLAB_TRAINING.md`.

In [ ]:
from pathlib import Path
import zipfile, os, subprocess, sys, json
archive = Path('/content/finet_colab_training.zip')
assert archive.is_file(), 'Upload finet_colab_training.zip using the Files sidebar first.'
PROJECT = Path('/content/boundary-aware-finet')
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert not name.startswith('/') and '..' not in Path(name).parts, 'Unsafe archive path'
    z.extractall(PROJECT)
os.chdir(PROJECT)
def run_checked(command):
    completed = subprocess.run(command, capture_output=True, text=True)
    print(completed.stdout); print(completed.stderr[-5000:])
    completed.check_returncode()
    return completed
run_checked([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'])
run_checked([sys.executable, '-m', 'pip', 'check'])

In [ ]:
import torch, torchvision
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), 'No CUDA GPU. Stop; request a GPU runtime. Do not start full training on CPU.'
free, total = torch.cuda.mem_get_info()
print({'gpu': torch.cuda.get_device_name(0), 'total_GiB': total/2**30, 'free_GiB': free/2**30})
run_checked([sys.executable, 'scripts/verify_backbone.py'])
run_checked([sys.executable, '-m', 'pytest', '-q', '--junitxml=outputs/colab_pytest.xml'])

## Mount Drive and obtain the verified official training pairs
Authorize Drive yourself when prompted. The helper downloads the SINet-V2-linked archive, checks the locally verified SHA-256, extracts original image/mask bytes, audits 3,040 COD10K + 1,000 CAMO pairs and creates the fixed 3,636/404 split.

Two original CAMO masks contain grayscale values: preserve them for structure loss. Two duplicate-image pairs stay together in one split. Data is extracted in `/content`; its archive is backed up once under `MyDrive/FINet/data/`. Checkpoints and the split remain in Drive. Final test datasets are separate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
DRIVE_ROOT = Path('/content/drive/MyDrive/FINet')
DATA_ROOT = Path('/content/finet_data/TrainDataset')
SPLIT = DRIVE_ROOT / 'splits/cod_camo_seed123456.json'
DATA_ARCHIVE = DRIVE_ROOT / 'data/COD-TrainDataset.zip'
DATA_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
local_archive = Path('/content/COD-TrainDataset.zip')
if DATA_ARCHIVE.is_file() and not local_archive.is_file():
    shutil.copy2(DATA_ARCHIVE, local_archive)
if not SPLIT.is_file() or not (DATA_ROOT/'Imgs').is_dir():
    run_checked([sys.executable, 'scripts/prepare_official_data.py', '--archive', str(local_archive), '--data-root', str(DATA_ROOT), '--manifest', str(SPLIT), '--download'])
from src.training_data import load_manifest, file_hash
assert file_hash(local_archive) == '255146828426e144c54867879824bcef27075b9e2386899fa0220f77551dc7be', 'Archive differs from verified release'
manifest = load_manifest(SPLIT, DATA_ROOT)
assert manifest['fingerprint'] == '2100aa40c52f01442d59b28973fee653eec655e370c8b1c1ff3c8d40cf61d724'
if not DATA_ARCHIVE.exists():
    shutil.copy2(local_archive, DATA_ARCHIVE)
(DRIVE_ROOT/'code').mkdir(exist_ok=True)
shutil.copy2(archive, DRIVE_ROOT/'code/finet_colab_training.zip')
print({'counts': manifest['source_counts'], 'train':len(manifest['train_ids']), 'validation':len(manifest['validation_ids']), 'split':manifest['fingerprint'], 'archive_backed_up':DATA_ARCHIVE.exists()})

## Measured smoke configuration
Start with batch 4, FP32, size 384, Adam lr 2.6e-4, weight decay 4e-8, original 200-epoch cosine schedule. The smoke performs only three updates plus one reload-continuation update, regardless of the schedule budget. If GPU memory is insufficient, choose batch 2 and a fresh smoke directory and remeasure. Never use batch 1. Batch 8 is optional only after measurement and requires a new smoke report.

Set `RUN_NAME` to a new experiment name; the runner refuses to overwrite existing smoke evidence. Keep this name/configuration when reconnecting to resume the same baseline.

In [ ]:
RUN_NAME = 'baseline_seed123456_b4_v1'
SMOKE_TAG = 'smoke_v1'  # Change only this tag to remeasure after a GPU-type change.
BATCH_SIZE = 4
SMOKE_DIR = DRIVE_ROOT / 'experiments' / (RUN_NAME + '_' + SMOKE_TAG)
RUN_DIR = DRIVE_ROOT / 'experiments' / RUN_NAME
cfg = json.loads(Path('configs/baseline.json').read_text())
cfg.update(data_root=str(DATA_ROOT), split_manifest=str(SPLIT), output_dir=str(SMOKE_DIR), device='cuda:0', batch_size=BATCH_SIZE, num_workers=2)
if (RUN_DIR/'initialization.pth').exists():
    cfg['initial_checkpoint'] = str(RUN_DIR/'initialization.pth')
SMOKE_CONFIG = PROJECT / 'outputs/colab_smoke_config.json'
SMOKE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
SMOKE_CONFIG.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))
REPORT = SMOKE_DIR / 'smoke_report.json'
if not REPORT.exists():
    subprocess.run([sys.executable, 'train_baseline.py', '--config', str(SMOKE_CONFIG), '--mode', 'smoke', '--steps', '3'], check=True)
proof = json.loads(REPORT.read_text())
assert proof['status'] == 'passed'
print(json.dumps(proof, indent=2))

In [ ]:
from src.training import TrainingConfig, run_signature
assert proof['run_signature'] == run_signature(TrainingConfig(**cfg), manifest), 'Configuration changed; use a new smoke run.'
assert proof['hardware'].get('gpu') == torch.cuda.get_device_name(0), 'GPU changed; remeasure smoke.'
step_seconds = proof['measured_steady_step_seconds']
train_steps = len(manifest['train_ids']) // BATCH_SIZE
validation = proof['validation_diagnostic']
estimated_train_s = train_steps * step_seconds
estimated_val_s = len(manifest['validation_ids']) * validation['seconds'] / validation['samples']
print({'measured_steady_step_seconds':step_seconds, 'steps_per_epoch':train_steps, 'approx_epoch_minutes_excluding_checkpoint_io':(estimated_train_s+estimated_val_s)/60, 'peak_allocated_GiB':proof['hardware']['peak_allocated_bytes']/2**30, 'peak_reserved_GiB':proof['hardware']['peak_reserved_bytes']/2**30})
print('A short-run extrapolation, not a promised duration. Drive checkpoint I/O, full validation, warm-up, contention and Colab disconnections add variability.')
cfg.update(output_dir=str(RUN_DIR), initial_checkpoint=str(SMOKE_DIR/'initialization.pth'))
FULL_CONFIG = PROJECT / 'outputs/colab_full_config.json'
FULL_CONFIG.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))

## STOP: inspect the evidence before full training
Check actual losses, finite gradients, encoder/head updates, checkpoint/optimizer/scheduler reload equality, free/peak GPU memory and the measured duration estimate. The initial full-model state saved before smoke is reused for the baseline; retain it for the later boundary ablation. No segmentation quality claim follows from a handful of steps.

Full training below is disabled. After reviewing the tested configuration and confirming GPU availability, deliberately set `RUN_FULL_TRAINING=True` to start. This is the long-running action. The latest epoch boundary, selected best validation MAE checkpoint, optimizer, scheduler, RNG state, logs and sample validation outputs are saved to Drive. Interrupted epochs restart from the last completed saved epoch. No mid-epoch exact resume claim is made.

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    assert torch.cuda.is_available()
    if 'training_job' in globals() and training_job.poll() is None:
        print('The existing training process is still running:', training_job.pid)
    else:
        command = [sys.executable, '-u', 'train_baseline.py', '--config', str(FULL_CONFIG), '--mode', 'train', '--start-full-training', '--verified-smoke-report', str(REPORT)]
        latest = RUN_DIR / 'last.pth'
        if latest.exists():
            command += ['--resume', str(latest)]
        # Keep the output folder empty for the trainer's first-run guard.
        TRAIN_STDOUT = DRIVE_ROOT / 'experiments' / (RUN_NAME + '_console.log')
        TRAIN_STDOUT.parent.mkdir(parents=True, exist_ok=True)
        with TRAIN_STDOUT.open('a', encoding='utf-8') as console:
            training_job = subprocess.Popen(command, stdout=console, stderr=subprocess.STDOUT, cwd=PROJECT)
        print({'pid':training_job.pid, 'console_log':str(TRAIN_STDOUT), 'note':'Process launched; inspect the monitor cell to verify actual training.'})
else:
    print('Full training not started. Review the measured configuration first.')

In [ ]:
def tail_text(path, limit=16384):
    path = Path(path)
    if not path.is_file():
        return '(not created yet)'
    with path.open('rb') as stream:
        stream.seek(0, 2)
        stream.seek(max(0, stream.tell()-limit))
        return stream.read().decode('utf-8', errors='replace')
if 'training_job' in globals():
    print({'pid':training_job.pid, 'return_code':training_job.poll()})
    print('Latest training records:', tail_text(RUN_DIR/'steps.jsonl', 5000))
    print('Completed epoch records:', tail_text(RUN_DIR/'epochs.jsonl', 5000))
    print('Console:', tail_text(TRAIN_STDOUT, 5000))
    print({'last_checkpoint_exists':(RUN_DIR/'last.pth').is_file(), 'best_checkpoint_exists':(RUN_DIR/'best.pth').is_file()})
else:
    print('No training process was launched from this notebook session.')

## After actual baseline training
Use `RUN_DIR/best.pth`, selected by held-out validation MAE. Evaluate on unseen labelled test data with `evaluation_colab.ipynb`; validation MAE is not a published benchmark result. The existing `predict.py` accepts this full project checkpoint and reports its project-trained provenance. It rejects initialization and smoke checkpoints.

Example after training: `python predict.py --image /content/unseen_camouflage.jpg --checkpoint <RUN_DIR>/best.pth --output-dir /content/unseen_result --device cuda:0`. This is intentionally not executed by this notebook. Use `scripts/prepare_experiments.py` for matched baseline/boundary configurations with the same saved full initialization. `app.py` provides the Gradio interface once genuine trained checkpoints are installed.